# 10 — More training images: 169 against 79, on four setups no fine-tuned model has read

**What this runs (GPU: about 2 hours on an A100; longer on an L4, not measured):** the approved fine-tuned model (79 training images) reads the 31
mCellSeg images outside the four setups it trained on. Then four new models are fine-tuned with the same recipe on
169, 41, 42 and 120 images of those four setups, and each reads the same 31. The rules and the training sets were
committed before this runs (`results/confluency_more_images.md`, `results/confluency_more_images_plan.json`).
Nothing is scored here: scoring runs once on the Mac.

**Before running:** upload `confluency_v6_bundle.zip` to `MyDrive/cultureqc/staged/`. The approved fine-tuned
weights are already there from nb/08 (`mcellseg_ftF_r2.weights`).

**I/O rule.** Nothing is read through the Drive mount except single copies to local disk (the bundle, the weights,
and the results zip if you are resuming). After each model, one small results zip is written back to Drive; the
169-image model's weights are copied to Drive once, in case it is ever proposed.

**If the session drops:** reconnect and run all cells again. Models whose readings are already in the results zip on
Drive are skipped, so at most one model is lost.

**Runtime:** L4 is fine and costs fewer credits per hour than A100; it is slower (not measured here). Turn on
**High-RAM**: the 169-image run holds 10.7 GB of training outlines in memory, plus the images, more than a
standard runtime has. The first cells check this. T4 is not recommended: it has no native bfloat16, which
Cellpose-SAM reads in, and it is the slowest.

**Short on credits?** Sections 1 and 2 answer the main question. You can stop after section 2 and tell me: the
curve (section 3) is then reported as not run.

**Bring back:** `MyDrive/cultureqc/staged/confluency_v6_results.zip` (under 1 MB).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, time, subprocess, json, hashlib, zipfile
DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'
STAGE_DIR = f'{DRIVE_ROOT}/staged'
BUNDLE = f'{STAGE_DIR}/confluency_v6_bundle.zip'
WEIGHTS = f'{STAGE_DIR}/mcellseg_ftF_r2.weights'
RESULTS_ZIP = f'{STAGE_DIR}/confluency_v6_results.zip'
N169_WEIGHTS = f'{STAGE_DIR}/mcellseg_more_n169.weights'
EXPECTED_COMMIT = 'e2df3b3'        # the commit the bundle was made from
REPO = '/content/cultureqc'
assert os.path.exists(BUNDLE), f'upload the bundle first: {BUNDLE}'
assert os.path.exists(WEIGHTS), f'the approved fine-tuned weights from nb/08 are missing: {WEIGHTS}'
print(f'{BUNDLE}: {os.path.getsize(BUNDLE)/1e6:.1f} MB; weights {os.path.getsize(WEIGHTS)/1e9:.2f} GB')

In [ ]:
t0 = time.time()
shutil.rmtree(REPO, ignore_errors=True)
shutil.copy(BUNDLE, '/content/bundle.zip')                 # one sequential read from Drive
!unzip -q -o /content/bundle.zip -d $REPO
os.remove('/content/bundle.zip')
%cd $REPO
commit = open('COMMIT').read().strip()
print('bundle commit', commit, f'— unpacked in {time.time()-t0:.0f} s')
assert commit.startswith(EXPECTED_COMMIT), (commit, EXPECTED_COMMIT)

In [ ]:
!pip install -q cellpose==4.2.1.1 tifffile scikit-image opencv-python-headless pyyaml matplotlib
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU (L4 or A100)'
import psutil
ram = psutil.virtual_memory().total / 1e9
print(f'system RAM {ram:.0f} GB')
assert ram >= 30, 'not enough memory for the 169-image run: Runtime > Change runtime type > High-RAM'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Every model here starts from the shipped Cellpose-SAM weights: Colab's download must be the approved file
from cellpose import models
_ = models.CellposeModel(gpu=True)
w = os.path.join(str(models.MODEL_DIR), 'cpsam_v2')
h = hashlib.sha256(open(w, 'rb').read()).hexdigest()
print('cpsam_v2', h)
assert h == '0f1cc3f7ecdd8a037a57c6c48d9d8921391be4cbce3fa9f13c3e3a2e1253c667', 'not the shipped weights: stop and tell me'
# The approved fine-tuned weights: one copy from Drive to local disk
t0 = time.time()
dst = 'cache/finetune/mcellseg_ftF_r2/models/mcellseg_ftF_r2'
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(WEIGHTS, dst)
h = hashlib.sha256(open(dst, 'rb').read()).hexdigest()
print('mcellseg_ftF_r2', h, f'— copied in {time.time()-t0:.0f} s')
assert h == '2d1549682c1da4829e7766bebf6e90e2373aa6c439f5138b098a71259d1239a2', 'not the approved fine-tuned weights: stop and tell me'
# The files that fix the rules and the training sets are byte-identical to the commit
SEALED = {
 "scripts/confluency_more_images.py": "7f1a2980b8a203fa9bb01c9d01a0f279f8d59d9474f0a7e5f8f054108ca57993",
 "results/confluency_more_images_plan.json": "6286e238a08096bb58f6cefe42f45e944f896902363c93bd596d35df3b9f041b",
 "results/confluency_more_images.md": "6d1cc197e1a58cf17cc7968c38b402143a4a395fa693230855d5e94b22c006e4",
 "scripts/confluency_mcellseg.py": "fd236fe2d51f8d1cd777450e4c6e257b6b44fa1a768428859c7f20c11df9f084",
 "scripts/confluency_finetune.py": "0bce35b023ad3c162d376ca08aa5974c37e092a5881caa27942af41c72c04744",
 "results/confluency_mcellseg_split.json": "84f6fb7e4eb99bdefc5f07ce7aeace8f7d0bfb68ff8f4bd7422c8348ef029c15"
}
for f, s in SEALED.items():
    assert hashlib.sha256(open(f, 'rb').read()).hexdigest() == s, f
print('committed files match', EXPECTED_COMMIT)

In [ ]:
# mCellSeg (Alam et al. 2026, CC BY 4.0) straight from Zenodo to local disk, MD5 from the record
t0 = time.time()
os.makedirs('/content/dl', exist_ok=True)
!curl -sL -o /content/dl/mCellSeg.zip "https://zenodo.org/api/records/20174259/files/mCellSeg.zip/content"
md5 = hashlib.md5(open('/content/dl/mCellSeg.zip', 'rb').read()).hexdigest()
print('md5', md5)
assert md5 == 'ed06d6e4c10e93b81703984cef852246'
os.makedirs('data/sources/mcellseg', exist_ok=True)
!unzip -q -o /content/dl/mCellSeg.zip -d data/sources/mcellseg
os.remove('/content/dl/mCellSeg.zip')
!ls data/sources/mcellseg/mCellSeg/labeled/images | wc -l
print(f'{time.time()-t0:.0f} s')
# Masks only: the split must match the committed one
!python scripts/confluency_mcellseg.py count

### ⏸ Checkpoint
Above, you should see **`committed files match`** and **`split matches the committed
results/confluency_mcellseg_split.json`**. If either is missing, stop and send me the output.

In [ ]:
CURVES = 'results/confluency_more_images_curves.json'
os.makedirs('logs', exist_ok=True)

def save_results(label):
    # one small zip to Drive: the readings (with each model's training manifest inside) + logs; no images, no weights
    keep = [CURVES] + [os.path.join('logs', f) for f in sorted(os.listdir('logs'))]
    local = '/content/confluency_v6_results.zip'
    with zipfile.ZipFile(local, 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('COMMIT', commit)
        for p in keep:
            if os.path.exists(p):
                z.write(p)
    shutil.copy(local, RESULTS_ZIP)
    print(label, '->', RESULTS_ZIP, f'{os.path.getsize(local)/1e6:.2f} MB')

# Resuming after a dropped session: bring back the readings already on Drive
done = set()
if os.path.exists(RESULTS_ZIP):
    with zipfile.ZipFile(RESULTS_ZIP) as z:
        if z.read('COMMIT').decode().strip() == commit and CURVES in z.namelist():
            z.extract(CURVES, '.')
            done = set(json.load(open(CURVES))['maps'])
print('already read:', sorted(done) or 'nothing')

def run(cmd, log):
    t0 = time.time()
    r = subprocess.run(['bash', '-c', f'set -o pipefail; {cmd} 2>&1 | tee -a logs/{log}.log | (grep -v -E "it/s]|s/it]" || true)'])
    print(f'[{log}] {(time.time()-t0)/60:.1f} min, exit {r.returncode}')
    if r.returncode != 0:
        save_results(f'{log} FAILED (partial results)')
        raise RuntimeError(f'{log} failed — send me logs/{log}.log (it is in the results zip)')

## 1 — The approved model reads the 31 (no training, a few minutes)

In [ ]:
if 'mcellseg_ftF_r2' not in done:
    run('python scripts/confluency_more_images.py read --run mcellseg_ftF_r2', 'read_mcellseg_ftF_r2')
    save_results('approved model read')

## 2 — Train on all 169, then read the 31 (about 50 min on A100)

In [ ]:
RUN = 'mcellseg_more_n169'
if RUN not in done:
    run(f'python scripts/confluency_more_images.py train --run {RUN}', f'train_{RUN}')
    run(f'python scripts/confluency_more_images.py read --run {RUN}', f'read_{RUN}')
    save_results(f'{RUN} read')
    man = json.load(open(f'cache/finetune/{RUN}/manifest.json'))
    t0 = time.time()
    shutil.copy(man['weights'], N169_WEIGHTS)              # one sequential write, kept in case it is ever proposed
    print('weights', man['weights_sha256'], '->', N169_WEIGHTS, f'{time.time()-t0:.0f} s')

## 3 — The curve: 41, 42 and 120 images (about 1 hour on A100)

In [ ]:
for RUN in ['mcellseg_more_n40_s0', 'mcellseg_more_n40_s1', 'mcellseg_more_n120_s0']:
    if RUN in done:
        print(RUN, 'already read'); continue
    run(f'python scripts/confluency_more_images.py train --run {RUN}', f'train_{RUN}')
    run(f'python scripts/confluency_more_images.py read --run {RUN}', f'read_{RUN}')
    save_results(f'{RUN} read')
print(sorted(json.load(open(CURVES))['maps']))

## Done
You should see five runs listed just above. Download `MyDrive/cultureqc/staged/confluency_v6_results.zip` and
tell me where it is (Downloads is fine). If your browser saves it as `confluency_v6_results(1).zip`, tell me that
name.